# T06: WoE scorecard และโมเดลคู่แข่ง (gradient boosting) บนข้อมูล Telecom

ข้อมูล: `data/sample_data_intw.csv` (Telecom micro-credit, อินโดนีเซีย) | feature: [`features/telecom.py`](../features/telecom.py) จาก T05

**โจทย์:** สร้าง WoE logistic scorecard และโมเดล gradient boosting เป็นคู่แข่ง (challenger) จาก feature มือถือ แบ่งข้อมูลตามเวลา แล้วรายงาน Gini และ KS บนข้อมูล test เทียบกับ AUC 0.62 (Gini 0.24) ของ R1

| ขั้น | ทำอะไร |
|---|---|
| 1 | โหลดข้อมูล สร้าง feature และเลือกกลุ่มที่ใช้พัฒนาโมเดล |
| 2 | แบ่งตามเวลา: train = มิ.ย. 2016, test = ก.ค. 2016 |
| 3 | Binning, WoE และ IV (เรียนจาก train เท่านั้น) |
| 4 | คัดตัวแปร 3 ด่าน (IV, correlation, วิจารณญาณ) |
| 5 | Logistic regression บน WoE และแปลงเป็นคะแนน (scorecard) |
| 6 | Challenger: gradient boosting (`HistGradientBoostingClassifier`) |
| 7 | ตารางผล AUC / Gini / KS บน test เทียบกับ R1 |

ขั้น 3 ถึง 5 ใช้วิธีและฟังก์ชันแบบเดียวกับ [`reference_solution.py`](../reference_solution.py) และ [`th_altdata_scorecard.py`](../th_altdata_scorecard.py)

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "..")                      # ให้ import features/ จากรากโปรเจกต์ได้
from features.telecom import FEATURE_COLUMNS, build_features

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)


def section(title):
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)


def check(label, value, ok=None):
    """[ OK ] = ผ่าน, [WARN] = ต้องดู, ว่าง = ข้อมูลเฉยๆ"""
    status = {True: "[ OK ]", False: "[WARN]", None: "      "}[ok]
    print(f"  {status}  {label:<34} {value}")

## ขั้นที่ 1: โหลดข้อมูล สร้าง feature และเลือกกลุ่มพัฒนาโมเดล

เลือกแถวตาม data dictionary ([docs/data_dictionary_telecom.md](../docs/data_dictionary_telecom.md) ข้อ 2.3 และ 3):

| ตัด | เหตุผล |
|---|---|
| แถวที่ซ้ำกันทุกคอลัมน์ (Q11) | 1 แถว |
| เดือนสิงหาคม (Q1) | ไม่มีคนเสียเลยสักคน ผลยังไม่ครบกำหนด ไม่ใช่ "ดี" จริง |
| `cnt_loans30` = 0 (Q2) | ไม่ได้กู้ก็ผิดนัดไม่ได้ ทุกแถวเป็น label 1 |

`label` ในไฟล์ใช้ 1 = จ่ายคืน (ดี) ซึ่งกลับด้านจากที่วงการใช้ จึงสร้าง `bad = 1 - label`

feature สร้างจากไฟล์ดิบทั้งไฟล์ก่อนตัดแถว เพราะ `build_features` คิดทีละแถว ไม่ได้เรียนอะไรจากข้อมูล จึงไม่มีข้อมูลจาก test รั่วเข้า train

In [2]:
section("1.1 LOAD + FEATURES")
raw = pd.read_csv("../data/sample_data_intw.csv")
raw["date"] = pd.to_datetime(raw["pdate"], format="%d-%m-%Y")
feat = build_features(raw).join(raw[["msisdn", "date", "label", "cnt_loans30"]])
feat["bad"] = 1 - feat["label"]
check("rows in file", f"{len(raw):,}")
check("features from T05", f"{len(FEATURE_COLUMNS)}: {FEATURE_COLUMNS[:3]} ...")

section("1.2 POPULATION: rows used for development")
step0 = feat
step1 = step0[~raw.duplicated()]
step2 = step1[step1["date"] < "2016-08-01"]
step3 = step2[step2["cnt_loans30"] >= 1]
for name, part in [("all rows", step0), ("drop full duplicate (Q11)", step1),
                   ("drop August (Q1, no outcome yet)", step2), ("drop no-loan rows (Q2)", step3)]:
    check(name, f"{len(part):>8,} rows | bad rate {part['bad'].mean():.2%}")
dev = step3
check("matches data dictionary (168,289 / 15.55%)", f"{len(dev):,} / {dev['bad'].mean():.2%}",
      ok=len(dev) == 168_289)


  1.1 LOAD + FEATURES
          rows in file                       209,593
          features from T05                  10: ['recharge_count_30', 'recharge_count_90', 'recharge_amount_30'] ...

  1.2 POPULATION: rows used for development


          all rows                            209,593 rows | bad rate 12.48%
          drop full duplicate (Q11)           209,592 rows | bad rate 12.48%
          drop August (Q1, no outcome yet)    168,918 rows | bad rate 15.49%
          drop no-loan rows (Q2)              168,289 rows | bad rate 15.55%
  [ OK ]  matches data dictionary (168,289 / 15.55%) 168,289 / 15.55%


## ขั้นที่ 2: แบ่งข้อมูลตามเวลา (train = มิ.ย., test = ก.ค.)

**ทำไมเลือกแบ่งตามเวลา ไม่ใช่สุ่มแบบ stratified**

1. **ใกล้กับการใช้งานจริง** ธนาคารสร้างโมเดลจากข้อมูลอดีต แล้วใช้ให้คะแนนลูกค้าในอนาคต test ที่มาจากเดือนถัดไปจึงเป็นข้อสอบที่เหมือนของจริงที่สุด
2. **ข้อมูลมีวันที่ (`pdate`)** และมีผลครบ 2 เดือนเต็ม (มิ.ย. และ ก.ค.) จึงแบ่งได้ train 83k และ test 85k ซึ่งใหญ่พอทั้งคู่
3. **วัดความคงที่ข้ามเวลาได้** ถ้าลูกค้าหรือสภาพตลาดเปลี่ยนระหว่างเดือน การสุ่มแบ่งจะมองไม่เห็น เพราะ train กับ test มาจากช่วงเดียวกัน
4. **การสุ่มแบ่งกับข้อมูลนี้มีความเสี่ยง** ลูกค้าบางคนมีหลายแถว (Q6) ถ้าสุ่มแบ่งเป็นรายแถว ลูกค้าคนเดียวกันจะอยู่ทั้งใน train และ test ทำให้ผลดูดีเกินจริง การแบ่งตามเวลาไม่มีปัญหานี้ภายในเดือนเดียวกัน

**ข้อจำกัด:** 2 เดือนห่างกันไม่มาก และเดือน ก.ค. มีลูกค้าส่วนหนึ่งที่เคยอยู่ในเดือน มิ.ย. ด้วย กรณีนี้ปกติในการใช้งานจริง (ลูกค้าเก่ากลับมากู้ใหม่) จึงไม่ตัดออก แต่แสดงจำนวนไว้ด้านล่าง

In [3]:
section("2.1 SPLIT: train = June 2016, test = July 2016")
train = dev[dev["date"] < "2016-07-01"].copy()
test = dev[dev["date"] >= "2016-07-01"].copy()
for name, part in [("train (June)", train), ("test (July)", test)]:
    check(name, f"{len(part):,} rows | {part['msisdn'].nunique():,} customers | bad rate {part['bad'].mean():.2%}")
in_both = test["msisdn"].isin(train["msisdn"])
check("test rows whose customer is also in train", f"{in_both.sum():,} ({in_both.mean():.1%})")
print("  -> bins, WoE and IV are learned from TRAIN only; test stays unseen until step 7")


  2.1 SPLIT: train = June 2016, test = July 2016
          train (June)                       83,131 rows | 78,949 customers | bad rate 15.95%
          test (July)                        85,158 rows | 80,905 customers | bad rate 15.15%
          test rows whose customer is also in train 7,368 (8.7%)
  -> bins, WoE and IV are learned from TRAIN only; test stays unseen until step 7


## ขั้นที่ 3: Binning, WoE และ IV (จาก train เท่านั้น)

- ตัวแปรที่มีค่าไม่เกิน 7 แบบใช้ค่าของตัวเองเป็นกลุ่ม ตัวแปรต่อเนื่องตัดเป็น 5 กลุ่มตาม quintile ของ train
- **ค่าว่างได้กลุ่ม `missing` ของตัวเอง** ไม่เติมเป็น 0 ค่าว่างใน T05 มีความหมาย เช่น `days_since_last_recharge` ว่างแปลว่าไม่ได้เติมเงินเลยใน 90 วัน
- WoE = ln(%ดี / %เสีย) ค่าบวกแปลว่าปลอดภัยกว่าค่าเฉลี่ย ค่าลบแปลว่าเสี่ยงกว่า IV = Σ (%ดี − %เสีย) × WoE

In [4]:
def make_bins(series, n_bins=5):
    """จุดตัด quintile ของตัวแปรต่อเนื่อง (ไม่นับค่าว่าง) หรือ None ถ้ามีค่าไม่กี่แบบ"""
    s = series.dropna()
    if not pd.api.types.is_numeric_dtype(s) or s.nunique() <= 7:
        return None
    edges = s.quantile(np.linspace(0, 1, n_bins + 1)).unique()
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


def label_value(v):
    """ชื่อกลุ่มของตัวแปรที่ใช้ค่าตัวเอง: 3.0 -> '3', ค่าว่าง -> 'missing'"""
    if pd.isna(v):
        return "missing"
    if isinstance(v, (int, float, np.number)) and float(v).is_integer():
        return str(int(v))
    return str(v)


def assign_bins(series, edges):
    """ชื่อกลุ่มของแต่ละแถว (จุดตัดจาก train); ค่าว่างได้กลุ่ม 'missing' ของตัวเอง"""
    if edges is None:
        return series.map(label_value)
    return pd.cut(series, bins=edges).astype(str).where(series.notna(), "missing")


def bin_order(series, edges):
    """ลำดับกลุ่มสำหรับแสดงผล: น้อยไปมาก แล้วตามด้วย missing"""
    if edges is None:
        order = [label_value(v) for v in sorted(series.dropna().unique())]
    else:
        order = list(pd.cut(pd.Series([], dtype=float), bins=edges).cat.categories.astype(str))
    return order + (["missing"] if series.isna().any() else [])


def woe_table(bins, target, order):
    """จำนวนคน, อัตรา bad, WoE และ IV ของแต่ละกลุ่ม"""
    t = pd.DataFrame({"bin": bins, "bad": target}).groupby("bin")["bad"].agg(customers="count", bads="sum")
    t = t.reindex([o for o in order if o in t.index])
    t["goods"] = t["customers"] - t["bads"]
    t["bad_rate"] = t["bads"] / t["customers"]
    pct_good = (t["goods"] + 0.5) / (t["goods"].sum() + 0.5)
    pct_bad = (t["bads"] + 0.5) / (t["bads"].sum() + 0.5)
    t["woe"] = np.log(pct_good / pct_bad)          # บวก = ปลอดภัยกว่าค่าเฉลี่ย, ลบ = เสี่ยงกว่า
    t["iv"] = (pct_good - pct_bad) * t["woe"]
    return t.round(3)


def iv_strength(v):
    if v < 0.02:
        return "useless"
    if v < 0.1:
        return "weak"
    if v < 0.3:
        return "medium"
    if v < 0.5:
        return "strong"
    return "very strong - check"


def show_woe(col):
    """ตาราง WoE ที่อ่านง่าย"""
    t = woe_tables[col]
    out = pd.DataFrame({
        "rows": t["customers"].map("{:,}".format),
        "%goods": (t["goods"] / t["goods"].sum()).map("{:.1%}".format),
        "%bads": (t["bads"] / t["bads"].sum()).map("{:.1%}".format),
        "bad_rate": t["bad_rate"].map("{:.1%}".format),
        "woe": t["woe"].map("{:+.3f}".format),
        "iv": t["iv"].map("{:.3f}".format),
        "risk": [("safer " + "+" * min(int(abs(w) * 4) + 1, 8)) if w > 0 else ("riskier " + "-" * min(int(abs(w) * 4) + 1, 8))
                 for w in t["woe"]],
    })
    out.index = t.index
    out.index.name = col
    print(out.to_string())
    print(f"  {'':>12} total IV = {t['iv'].sum():.3f} ({iv_strength(t['iv'].sum())})")


def woe_direction(col):
    """นับว่า WoE เปลี่ยนทิศกี่ครั้ง (ไม่นับกลุ่ม missing): 0 = เรียงทางเดียว อธิบายง่าย"""
    w = woe_tables[col]["woe"].drop("missing", errors="ignore").to_numpy()
    signs = np.sign(np.diff(w))
    signs = signs[signs != 0]
    return int((np.diff(signs) != 0).sum()) if len(signs) > 1 else 0


candidates = FEATURE_COLUMNS
bin_edges, bin_orders, woe_tables = {}, {}, {}
for col in candidates:
    bin_edges[col] = make_bins(train[col])
    bin_orders[col] = bin_order(train[col], bin_edges[col])
    woe_tables[col] = woe_table(assign_bins(train[col], bin_edges[col]), train["bad"], bin_orders[col])

section("3.1 BINNING: how each variable was cut into groups")
for col in candidates:
    edges = bin_edges[col]
    method = "own values" if edges is None else f"quintiles {np.round(edges[1:-1], 1).tolist()}"
    extra = " + missing bin" if "missing" in woe_tables[col].index else ""
    check(col, f"{len(woe_tables[col])} bins | {method}{extra}")

section("3.2 IV RANKING")
iv = pd.Series({col: woe_tables[col]["iv"].sum() for col in candidates}).sort_values(ascending=False)
print("  scale: <0.02 useless | 0.02-0.1 weak | 0.1-0.3 medium | 0.3-0.5 strong | >0.5 very strong (check)\n")
for col, value in iv.items():
    print(f"  {col:<26} {value:6.3f}  {iv_strength(value):<20} {'#' * max(1, int(value * 40))}")


  3.1 BINNING: how each variable was cut into groups
          recharge_count_30                  4 bins | quintiles [1.0, 3.0, 5.0]
          recharge_count_90                  5 bins | quintiles [1.0, 2.0, 3.0, 6.0]
          recharge_amount_30                 4 bins | quintiles [1924.0, 4132.0, 8500.0]
          recharge_amount_90                 5 bins | quintiles [773.0, 2320.0, 4983.0, 9869.0]
          days_since_last_recharge           5 bins | quintiles [2.0, 4.0, 8.0] + missing bin
          prebal_median_30                   6 bins | quintiles [14.9, 29.0, 58.2, 121.0] + missing bin
          prebal_median_90                   6 bins | quintiles [15.5, 29.5, 58.0, 118.0] + missing bin
          loan_count_30                      3 bins | quintiles [2.0, 3.0]
          loan_count_90                      4 bins | quintiles [2.0, 3.0] + missing bin
          tenure_days                        6 bins | quintiles [181.0, 370.0, 631.0, 1049.0] + missing bin

  3.2 IV RANKING
  sc

In [5]:
section("3.3 WoE TABLES (train): which groups are safer / riskier")
print("  WoE = ln(%goods / %bads)   + = safer than average   - = riskier   0 = average")
for col in iv.index:
    print()
    show_woe(col)


  3.3 WoE TABLES (train): which groups are safer / riskier
  WoE = ln(%goods / %bads)   + = safer than average   - = riskier   0 = average

                      rows %goods  %bads bad_rate     woe     iv            risk
recharge_amount_90                                                              
(-inf, 773.0]       19,260  16.1%  60.5%    41.7%  -1.326  0.590  riskier ------
(773.0, 2320.0]     14,279  16.8%  19.2%    17.9%  -0.135  0.003       riskier -
(2320.0, 4983.0]    16,342  21.3%  11.0%     8.9%  +0.664  0.069       safer +++
(4983.0, 9869.0]    16,628  22.7%   6.0%     4.8%  +1.321  0.219    safer ++++++
(9869.0, inf]       16,622  23.2%   3.2%     2.6%  +1.970  0.393  safer ++++++++
               total IV = 1.274 (very strong - check)

                     rows %goods  %bads bad_rate     woe     iv            risk
recharge_count_90                                                              
(-inf, 1.0]        30,051  28.4%  76.9%    33.9%  -0.995  0.482    riskier --

## ขั้นที่ 4: คัดตัวแปร 3 ด่าน

1. ตัดตัวที่ IV < 0.02 (ไม่มีประโยชน์)
2. ตัดตัวที่ซ้ำกัน: ถ้า |correlation ของ WoE| > 0.7 ให้เก็บตัวที่ IV สูงกว่า
3. วิจารณญาณ: รูปแบบ WoE ต้องมีเหตุผลทางธุรกิจ ดูจากจำนวนครั้งที่ WoE เปลี่ยนทิศ

In [6]:
def to_woe(data, col):
    """แทนค่าจริงด้วย WoE ของกลุ่ม (จุดตัดและ WoE จาก train); กลุ่มที่ไม่เคยเห็น -> 0"""
    mapping = woe_tables[col]["woe"]
    return assign_bins(data[col], bin_edges[col]).map(mapping).astype(float).fillna(0)


section("4.1 GATE 1: drop useless variables (IV < 0.02)")
strong = [c for c in iv.index if iv[c] >= 0.02]
for col in iv.index:
    check(col, f"IV {iv[col]:.3f}", ok=iv[col] >= 0.02)
print(f"  -> dropped: {[c for c in iv.index if iv[c] < 0.02] or 'none'} | {len(strong)} variables pass")

section("4.2 GATE 2: drop duplicates (|correlation of WoE| > 0.7)")
train_woe = pd.DataFrame({c: to_woe(train, c) for c in strong})
corr = train_woe.corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(strong) for b in strong[i + 1:] if abs(corr.loc[a, b]) > 0.7]
to_drop = set()
for a, b, r in sorted(pairs, key=lambda p: -abs(p[2])):
    if a in to_drop or b in to_drop:
        continue
    loser, winner = (a, b) if iv[a] < iv[b] else (b, a)
    to_drop.add(loser)
    check(f"{a} vs {b}", f"corr {r:.2f} -> keep {winner} (IV {iv[winner]:.3f}), drop {loser}", ok=False)
if not pairs:
    print("  no pair above 0.7")
after_rules = [c for c in strong if c not in to_drop]
print(f"  -> {len(after_rules)} variables left: {after_rules}")

section("4.3 GATE 3: judgement - does the WoE pattern make business sense?")
print("  direction changes = how often WoE switches between going up and going down (0 = one direction)\n")
for col in after_rules:
    turns = woe_direction(col)
    check(col, f"{turns} direction change(s)", ok=turns <= 1)
JUDGEMENT_REASON = {}
for col, reason in JUDGEMENT_REASON.items():
    print()
    show_woe(col)
    print(f"  -> DROP: {reason}")
selected = [c for c in after_rules if c not in JUDGEMENT_REASON]

section("4.4 FUNNEL: candidates -> final features")
check("start (T05 features)", len(candidates))
check("after gate 1 (IV >= 0.02)", len(strong))
check("after gate 2 (no duplicates)", f"{len(after_rules)}  dropped {sorted(to_drop)}")
check("after gate 3 (judgement)", f"{len(selected)}  dropped {list(JUDGEMENT_REASON) or 'none'}")

X_train = pd.DataFrame({c: to_woe(train, c) for c in selected})
X_test = pd.DataFrame({c: to_woe(test, c) for c in selected})
y_train, y_test = train["bad"], test["bad"]


  4.1 GATE 1: drop useless variables (IV < 0.02)
  [ OK ]  recharge_amount_90                 IV 1.274
  [ OK ]  recharge_count_90                  IV 1.273
  [ OK ]  recharge_count_30                  IV 1.131
  [ OK ]  recharge_amount_30                 IV 1.079
  [ OK ]  prebal_median_30                   IV 0.875
  [ OK ]  prebal_median_90                   IV 0.868
  [ OK ]  days_since_last_recharge           IV 0.828
  [ OK ]  loan_count_30                      IV 0.644
  [ OK ]  loan_count_90                      IV 0.640
  [ OK ]  tenure_days                        IV 0.074
  -> dropped: none | 10 variables pass

  4.2 GATE 2: drop duplicates (|correlation of WoE| > 0.7)


  [WARN]  loan_count_30 vs loan_count_90     corr 1.00 -> keep loan_count_30 (IV 0.644), drop loan_count_90
  [WARN]  recharge_count_90 vs recharge_count_30 corr 0.94 -> keep recharge_count_90 (IV 1.273), drop recharge_count_30
  [WARN]  recharge_amount_90 vs recharge_amount_30 corr 0.89 -> keep recharge_amount_90 (IV 1.274), drop recharge_amount_30
  [WARN]  prebal_median_30 vs prebal_median_90 corr 0.87 -> keep prebal_median_30 (IV 0.875), drop prebal_median_90
  [WARN]  recharge_amount_90 vs recharge_count_90 corr 0.81 -> keep recharge_amount_90 (IV 1.274), drop recharge_count_90
  [WARN]  prebal_median_30 vs days_since_last_recharge corr 0.70 -> keep prebal_median_30 (IV 0.875), drop days_since_last_recharge
  -> 4 variables left: ['recharge_amount_90', 'prebal_median_30', 'loan_count_30', 'tenure_days']

  4.3 GATE 3: judgement - does the WoE pattern make business sense?
  direction changes = how often WoE switches between going up and going down (0 = one direction)

  [ OK ]  rec

## ขั้นที่ 5: Logistic regression บน WoE และแปลงเป็นคะแนน

- target คือ `bad` WoE บวกแปลว่าปลอดภัย สัมประสิทธิ์ทุกตัวจึงต้องติดลบ ถ้าตัวไหนเป็นบวก ให้ตัดตัวที่ IV ต่ำสุดออกก่อน แล้วฝึกใหม่
- คะแนน: 600 = odds ดี:เสีย 50:1 และทุก 20 คะแนน (PDO) odds เพิ่มเป็น 2 เท่า คะแนนสูงแปลว่าปลอดภัย

In [7]:
from sklearn.linear_model import LogisticRegression


def show_coefs(model, columns):
    for col, c in zip(columns, model.coef_[0]):
        check(col, f"{c:+.3f}  {'correct sign' if c < 0 else 'WRONG SIGN'}", ok=c < 0)
    check("intercept", f"{model.intercept_[0]:+.3f}")


section("5.1 FIT + SIGN FIX: every coefficient must be negative")
logit = LogisticRegression(C=1e6, max_iter=1000)
logit.fit(X_train, y_train)
coefs = pd.Series(logit.coef_[0], index=X_train.columns)
show_coefs(logit, X_train.columns)
while (coefs > 0).any():
    worst = min(coefs[coefs > 0].index, key=lambda c: iv[c])
    print(f"  -> dropping {worst}: coefficient {coefs[worst]:+.3f} has the wrong sign (IV {iv[worst]:.3f})")
    selected = [c for c in selected if c != worst]
    X_train, X_test = X_train[selected], X_test[selected]
    logit.fit(X_train, y_train)
    coefs = pd.Series(logit.coef_[0], index=X_train.columns)

section("5.2 FINAL MODEL")
show_coefs(logit, X_train.columns)
check("all coefficients negative", bool((coefs < 0).all()), ok=bool((coefs < 0).all()))
b0 = logit.intercept_[0]
print(f"\n  average row (all WoE = 0): PD = {1 / (1 + np.exp(-b0)):.1%}  (actual train bad rate {y_train.mean():.1%})")


  5.1 FIT + SIGN FIX: every coefficient must be negative
  [ OK ]  recharge_amount_90                 -0.635  correct sign
  [ OK ]  prebal_median_30                   -0.371  correct sign
  [ OK ]  loan_count_30                      -0.450  correct sign
  [ OK ]  tenure_days                        -0.778  correct sign
          intercept                          -1.658

  5.2 FINAL MODEL
  [ OK ]  recharge_amount_90                 -0.635  correct sign
  [ OK ]  prebal_median_30                   -0.371  correct sign
  [ OK ]  loan_count_30                      -0.450  correct sign
  [ OK ]  tenure_days                        -0.778  correct sign
          intercept                          -1.658
  [ OK ]  all coefficients negative          True

  average row (all WoE = 0): PD = 16.0%  (actual train bad rate 16.0%)


In [8]:
section("5.3 SCORECARD: points for each group (600 = odds 50:1, PDO 20)")
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
n = len(selected)
base_share = (offset - factor * b0) / n
base_points = round(base_share)
print(f"  factor {factor:.2f} | offset {offset:.2f} | points = {base_share:.1f} - {factor:.2f} x coefficient x WoE\n")
rows = []
for col in selected:
    for bin_label, woe in woe_tables[col]["woe"].items():
        rows.append({"feature": col, "bin": bin_label, "woe": woe,
                     "points": round(base_share - factor * coefs[col] * woe)})
scorecard = pd.DataFrame(rows)
for col in selected:
    part = scorecard[scorecard["feature"] == col]
    print(f"  {col}  (coefficient {coefs[col]:+.3f}, spread {part['points'].max() - part['points'].min()} points)")
    for _, r in part.iterrows():
        print(f"      {r['bin']:<22} WoE {r['woe']:+.3f}  -> {int(r['points']):>4} pts  {'#' * max(1, (int(r['points']) - 40) // 2)}")
    print()


def score(data):
    """คะแนนรวมจากตาราง scorecard (กลุ่มที่ไม่เคยเห็นใน train ได้คะแนนพื้นฐาน)"""
    pts = {}
    for col in selected:
        lookup = scorecard[scorecard["feature"] == col].set_index("bin")["points"]
        pts[col] = assign_bins(data[col], bin_edges[col]).map(lookup).fillna(base_points).astype(float)
    return pd.DataFrame(pts, index=data.index).sum(axis=1)


def pd_from_score(s):
    return 1 / (1 + np.exp((s - offset) / factor))


section("5.4 VERIFY: scorecard gives the same answer as the model")
train_score, test_score = score(train), score(test)
gap = np.abs(pd_from_score(train_score) - logit.predict_proba(X_train)[:, 1]).max()
check("score range (train)", f"{int(train_score.min())} to {int(train_score.max())}")
check("max difference in PD vs model", f"{gap:.4f} (from rounding points)", ok=gap < 0.02)


  5.3 SCORECARD: points for each group (600 = odds 50:1, PDO 20)
  factor 28.85 | offset 487.12 | points = 133.7 - 28.85 x coefficient x WoE

  recharge_amount_90  (coefficient -0.635, spread 61 points)
      (-inf, 773.0]          WoE -1.326  ->  109 pts  ##################################
      (773.0, 2320.0]        WoE -0.135  ->  131 pts  #############################################
      (2320.0, 4983.0]       WoE +0.664  ->  146 pts  #####################################################
      (4983.0, 9869.0]       WoE +1.321  ->  158 pts  ###########################################################
      (9869.0, inf]          WoE +1.970  ->  170 pts  #################################################################

  prebal_median_30  (coefficient -0.371, spread 29 points)
      (-inf, 14.93]          WoE +0.037  ->  134 pts  ###############################################
      (14.93, 29.0]          WoE +0.774  ->  142 pts  #################################################

          score range (train)                482 to 621
  [ OK ]  max difference in PD vs model      0.0093 (from rounding points)


## ขั้นที่ 6: Challenger: gradient boosting

`HistGradientBoostingClassifier` จาก scikit-learn (มีใน `requirements.txt` อยู่แล้ว):

- สร้างต้นไม้เล็ก ๆ ต่อกันหลายต้น แต่ละต้นแก้ส่วนที่ต้นก่อนหน้าทายผิด จึงจับความสัมพันธ์ที่ไม่เป็นเส้นตรงและ interaction ระหว่างตัวแปรได้เอง
- **ใช้ feature ดิบจาก T05 ครบทั้ง 10 ตัว** ไม่ bin ไม่ทำ WoE ไม่คัดตัวแปร เพื่อวัดว่าข้อมูลชุดนี้มีสัญญาณทั้งหมดเท่าไร scorecard จะได้เทียบกับเพดานนี้
- รับค่าว่างได้เอง (เรียนเองว่าค่าว่างควรไปทางไหน)
- ใช้ค่าตั้งต้นของ scikit-learn ไม่ได้ปรับ hyperparameter เอง ระบบจะกันข้อมูล 10% ของ **train** ไว้เป็น validation เพื่อหยุดฝึกเมื่อผลไม่ดีขึ้น (early stopping) test จึงไม่ได้ถูกใช้ระหว่างฝึก

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier

section("6.1 FIT: gradient boosting on the raw T05 features (train)")
gbm = HistGradientBoostingClassifier(random_state=42)
gbm.fit(train[FEATURE_COLUMNS], y_train)
check("features used", len(FEATURE_COLUMNS))
check("trees built (early stopping)", gbm.n_iter_)
gbm_train = gbm.predict_proba(train[FEATURE_COLUMNS])[:, 1]   # PD: สูง = เสี่ยง
gbm_test = gbm.predict_proba(test[FEATURE_COLUMNS])[:, 1]


  6.1 FIT: gradient boosting on the raw T05 features (train)


          features used                      10
          trees built (early stopping)       94


## ขั้นที่ 7: ตารางผล: AUC / Gini / KS บน test (ก.ค.) เทียบกับ R1

- **AUC** คือโอกาสที่โมเดลให้คนเสียมีความเสี่ยงสูงกว่าคนดี เมื่อสุ่มมาเทียบกันคนละ 1 คน **Gini** = 2 × AUC − 1 **KS** คือระยะห่างที่มากที่สุดระหว่างกราฟสะสมของคนดีกับคนเสีย
- scorecard ใช้คะแนน (คะแนนต่ำแปลว่าเสี่ยง) จึงคำนวณจาก `-score` ส่วน gradient boosting ใช้ PD (สูงแปลว่าเสี่ยง)
- **ผลหลักคือแถว test** แถว train ใช้ดู overfitting เท่านั้น
- R1 (Ots, Liiv และ Tur 2020) รายงานแค่ AUC ไม่มี KS

In [10]:
from sklearn.metrics import roc_auc_score, roc_curve


def metrics(y, risk):
    """AUC/Gini/KS; risk = ค่าที่ยิ่งสูงยิ่งเสี่ยง"""
    fpr, tpr, _ = roc_curve(y, risk)
    auc = roc_auc_score(y, risk)
    return {"AUC": auc, "Gini": 2 * auc - 1, "KS": (tpr - fpr).max()}


R1_AUC = 0.62
rows = {
    ("WoE logistic scorecard", "test (July)"): metrics(y_test, -test_score),
    ("Gradient boosting (challenger)", "test (July)"): metrics(y_test, gbm_test),
    ("R1 benchmark (Ots et al. 2020)", "paper"): {"AUC": R1_AUC, "Gini": 2 * R1_AUC - 1, "KS": np.nan},
    ("WoE logistic scorecard", "train (June)"): metrics(y_train, -train_score),
    ("Gradient boosting (challenger)", "train (June)"): metrics(y_train, gbm_train),
}
results = pd.DataFrame(rows).T
results.index.names = ["model", "data"]

section("7.1 RESULTS TABLE")
print(results.round(3).to_string(na_rep="-"))

sc = results.loc[("WoE logistic scorecard", "test (July)")]
gb = results.loc[("Gradient boosting (challenger)", "test (July)")]
r1_gini = 2 * R1_AUC - 1

section("7.2 COMPARISON on test")
check("scorecard Gini vs R1 (0.24)", f"{sc['Gini']:.3f}  ({sc['Gini'] - r1_gini:+.3f})", ok=sc["Gini"] > r1_gini)
check("challenger Gini vs R1 (0.24)", f"{gb['Gini']:.3f}  ({gb['Gini'] - r1_gini:+.3f})", ok=gb["Gini"] > r1_gini)
check("challenger - scorecard: Gini", f"{gb['Gini'] - sc['Gini']:+.3f}")
check("challenger - scorecard: KS", f"{gb['KS'] - sc['KS']:+.3f}")

section("7.3 OVERFITTING: Gini train - test")
for m in ["WoE logistic scorecard", "Gradient boosting (challenger)"]:
    g = results.loc[(m, "train (June)"), "Gini"] - results.loc[(m, "test (July)"), "Gini"]
    check(m, f"{g:+.3f}", ok=abs(g) < 0.05)


  7.1 RESULTS TABLE
                                               AUC   Gini     KS
model                          data                             
WoE logistic scorecard         test (July)   0.833  0.667  0.523
Gradient boosting (challenger) test (July)   0.843  0.685  0.538
R1 benchmark (Ots et al. 2020) paper         0.620  0.240      -
WoE logistic scorecard         train (June)  0.813  0.627  0.493
Gradient boosting (challenger) train (June)  0.863  0.726  0.587

  7.2 COMPARISON on test
  [ OK ]  scorecard Gini vs R1 (0.24)        0.667  (+0.427)
  [ OK ]  challenger Gini vs R1 (0.24)       0.685  (+0.445)
          challenger - scorecard: Gini       +0.018
          challenger - scorecard: KS         +0.015

  7.3 OVERFITTING: Gini train - test
  [ OK ]  WoE logistic scorecard             -0.040
  [ OK ]  Gradient boosting (challenger)     +0.041


## สรุป

### ตารางผลบน test (ก.ค. 2016, 85,158 แถว)

| โมเดล | ตัวแปร | AUC | Gini | KS |
|---|---|---|---|---|
| WoE logistic scorecard | 4 | 0.833 | 0.667 | 0.523 |
| Gradient boosting (challenger) | 10 | 0.843 | 0.685 | 0.538 |
| R1 (Ots, Liiv และ Tur 2020) | 22 | 0.62 | 0.24 | – |

### อ่านผลอย่างไร

1. **ทั้งสองโมเดลสูงกว่า R1 มาก** Gini 0.67–0.69 เทียบกับ 0.24 แต่ไม่ใช่การเทียบแบบเท่ากันทุกด้าน R1 อ่านข้อมูลจากเครื่องโทรศัพท์ (จำนวนสาย SMS รูปภาพ) ของผู้กู้ 2,503 คนในยุโรป และชุดทดสอบมีราว 250 คน ส่วนข้อมูลชุดนี้เป็นข้อมูลการเติมเงินและการยืมค่าโทรจากค่ายมือถือ ผลิตภัณฑ์เป็นเงินกู้ค่าโทร 5 วัน และนิยามคนเสียคือ "ไม่คืนใน 5 วัน" สิ่งที่สรุปได้คือ ข้อมูลการเติมเงินจากค่ายมือถือมีสัญญาณแรงกว่าข้อมูลการใช้เครื่องแบบใน R1 มาก ซึ่งตรงกับข้อสรุปใน research notes
2. **Challenger ชนะ scorecard เพียงเล็กน้อย** Gini +0.018 และ KS +0.015 ทั้งที่ใช้ตัวแปรดิบครบ 10 ตัวและจับความสัมพันธ์ที่ไม่เป็นเส้นตรงได้ scorecard ที่ใช้แค่ 4 ตัวแปรและอธิบายเป็นตารางคะแนนได้ จับสัญญาณส่วนใหญ่ได้แล้ว
3. **Overfitting:** challenger มี Gini บน train สูงกว่า test 0.041 ส่วน scorecard ทำได้ดีบน test มากกว่า train (−0.040) ข้อนี้ไม่ใช่สัญญาณของ overfitting แต่แสดงว่าประชากรเดือน ก.ค. ต่างจาก มิ.ย. เล็กน้อย (อัตราคนเสีย 15.15% เทียบกับ 15.95%) ซึ่งการแบ่งตามเวลาเปิดให้เห็น

### เรื่องที่ยังเปิดอยู่ (ควรคุยกับพี่เลี้ยง)

- **IV ของ 9 จาก 10 ตัวแปรเกิน 0.5** ("very strong - check") ตามเกณฑ์ IV ที่สูงขนาดนี้ควรตรวจว่ามีข้อมูลรั่ว (leakage) หรือไม่ คำถามคือช่วง 30/90 วันของ feature ที่วัด ณ `pdate` รวมช่วง 5 วันหลังกู้ ซึ่งเป็นช่วงที่ลูกค้าเติมเงินเพื่อคืนหนี้ด้วยหรือไม่ data dictionary ยังไม่มีคำตอบ ต้องถามเจ้าของข้อมูล
- **Gate 3: `prebal_median_30` มี WoE เปลี่ยนทิศ 2 ครั้ง** ในช่วงกลาง (+0.77 → +0.52 → +0.64) สัญญาณหลักมาจากกลุ่ม `missing` (ไม่ได้เติมเงินใน 30 วัน) และกลุ่มบนสุด notebook นี้ยังเก็บตัวแปรนี้ไว้ ทางเลือกคือรวม bin ช่วงกลางเข้าด้วยกัน หรือตัดตัวแปรนี้ออก